# Field Shift · v3 soil water model with ERA5-Land's own rain

Rules written and committed before this notebook ran: [`docs/ML_V3_ERA5_RAIN.md`](../../docs/ML_V3_ERA5_RAIN.md) §2.
The tested v1 models are read-only. Writes `artifacts/v3_era5_rain/`.

## 1. Tables 1998–2025 with ERA5-Land rain

In [1]:
import sys, json, math, time, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, xgboost as xgb
warnings.filterwarnings("ignore", message="(All-NaN slice|Mean of empty slice)")
ML_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ML_DIR))
from fs_ml import data as D, features as F, evaluate as E, project as P, era5 as R
from fs_ml.predict import load

SEED, N_BOOT, NOMINAL = 42, 1000, 0.90
ART = ML_DIR / "artifacts" / "v3_era5_rain"; ART.mkdir(parents=True, exist_ok=True)
V1, ND = load("sm_root_monthly"), load("ndvi_quarterly")
SPLITS = {"training": (1998, 2017), "validation": (2018, 2022), "testing": (2023, 2025)}
CRIT = {"A": 0.20, "C": 0.04, "D": 0.01, "F": (0.87, 0.93)}

def build(load_years, suffix, keep):
    monthly, _ = D.load_gee_monthly(load_years); veg, _ = D.load_gee_quarterly(load_years)
    pw = {p: D.load_power(f"power_{p}{suffix}", load_years)[0] for p in ["t2m", "rh2m", "ws2m", "allsky"]}
    merra = pw["t2m"].merge(pw["rh2m"], on=["power_lat", "power_lon", "date"]).merge(pw["ws2m"], on=["power_lat", "power_lon", "date"])
    gaez = {k: D.load_gaez(k)[0] for k in F.GAEZ_FEATURES}
    cells = F.static_cells(D.cells_from_ids(monthly.cell_id), gaez, merra[["power_lat", "power_lon"]].drop_duplicates().reset_index(drop=True),
                           pw["allsky"][["power_lat", "power_lon"]].drop_duplicates().reset_index(drop=True))
    mp = F.build_monthly_panel(monthly, cells, merra, pw["allsky"]); qp = F.build_quarterly_panel(mp, veg)
    extra = pd.DataFrame({"cell_id": cells.cell_id.to_numpy(), "block": np.asarray(E.spatial_blocks(cells.lat, cells.lon)),
                          "province": np.asarray(D.provinces_of(cells))})
    return mp.merge(extra, on="cell_id")[lambda d: d.year.between(*keep)], qp.merge(extra, on="cell_id")[lambda d: d.year.between(*keep)]

parts = [build((1998, 2017), "", (1998, 2017)), build((2017, 2022), "_2018_2022", (2018, 2022)), build((2022, 2025), "_2023_2025", (2023, 2025))]
MP = pd.concat([p[0] for p in parts], ignore_index=True); QP = pd.concat([p[1] for p in parts], ignore_index=True)
MP = R.add_to_monthly(MP, R.load_era5_rain(MP.cell_id.unique()))
print(len(MP), "rows | ERA5 rain missing:", int(MP.era5_rain_mm.isna().sum()), "| cells:", MP.cell_id.nunique())

1058064 rows | ERA5 rain missing: 7392 | cells: 3149


## 2. Train v3 (same protocol and hyperparameters as v1, plus monotone constraints)

In [2]:
T = V1["card"]["target"]["column"]
FEATS = V1["features"] + R.FEATURES
MONO = {f: 1 for f in FEATS if "rain" in f} | {"pet_mm": -1, "t2m_c": -1}
base = {k: v for k, v in V1["card"]["hyperparameters"].items() if k != "n_estimators"} | {"n_estimators": 5000, "early_stopping_rounds": 100, "monotone_constraints": MONO}
tr = MP[MP.year.between(*SPLITS["training"]) & MP[T].notna()]
fit, es = tr[tr.year <= 2015], tr[tr.year.between(2016, 2017)]
t0 = time.time()
m1 = xgb.XGBRegressor(**base).fit(fit[FEATS], fit[T], eval_set=[(es[FEATS], es[T])], verbose=False)
best = m1.best_iteration + 1
params = {k: v for k, v in base.items() if k != "early_stopping_rounds"} | {"n_estimators": best}
V3m = xgb.XGBRegressor(**params).fit(tr[FEATS], tr[T], verbose=False)
V3 = {"model": V3m, "features": FEATS, "bounds": E.feature_bounds(tr[FEATS]), "card": V1["card"]}
print("trees", best, "(v1", V1["card"]["trees"], ") |", round(time.time() - t0), "s | monotone:", MONO)

trees 1877 (v1 1122 ) | 50 s | monotone: {'rain_mm': 1, 'rain_mm_lag1': 1, 'rain_mm_lag2': 1, 'rain_mm_sum6': 1, 'era5_rain_mm': 1, 'era5_rain_mm_lag1': 1, 'era5_rain_mm_lag2': 1, 'era5_rain_mm_sum6': 1, 'pet_mm': -1, 't2m_c': -1}


## 3. Validation 2018–2022 (R1, R2)

In [3]:
def rows(period):
    d = MP[MP.year.between(*SPLITS[period]) & MP[T].notna()].reset_index(drop=True)
    clim = E.predict_climatology(E.climatology(tr, T, ["cell_id", "month"], "month"), d)
    pers = np.where(d.sm_root_m3m3_lag1.isna(), clim, d.sm_root_m3m3_lag1)
    return d, d[T].to_numpy(), {"v3": V3m.predict(d[FEATS]), "v1": V1["model"].predict(d[V1["features"]]), "climatology": clim, "persistence": pers}

VAL = rows("validation"); d, y, p = VAL
CI = E.block_bootstrap(y, p, d.block.to_numpy(), "v3", N_BOOT, SEED).set_index(["model", "metric"])
sk, rmse, bias = CI.loc[("v3", "skill vs persistence")], CI.loc[("v3", "RMSE")], CI.loc[("v3", "bias")]
worst_year = min(E.skill(E.regression_metrics(y[d.year == yr], p["v3"][d.year == yr])["RMSE"], E.regression_metrics(y[d.year == yr], p["persistence"][d.year == yr])["RMSE"]) for yr in range(2018, 2023))
inside = d.province != "outside"
worst_prov = min(E.skill(E.regression_metrics(y[(d.province == pr).to_numpy()], p["v3"][(d.province == pr).to_numpy()])["RMSE"],
                          E.regression_metrics(y[(d.province == pr).to_numpy()], p["persistence"][(d.province == pr).to_numpy()])["RMSE"]) for pr in d.province[inside].unique())
res = p["v3"] - y; yr = d.year.to_numpy(); seas = np.where(d.month <= 4, "wet", "dry")
half = {s: E.conformal_halfwidth(res[(yr <= 2020) & (seas == s)], NOMINAL) for s in ("wet", "dry")}
chk = yr >= 2021; cov = float((np.abs(res[chk]) <= np.vectorize(half.get)(seas[chk])).mean())
R1 = [("A", f"skill vs persistence {sk.value:.3f} >= 0.20", sk.value >= CRIT["A"]), ("B", f"lower CI {sk.ci_low:.3f} > 0", sk.ci_low > 0),
      ("C", f"RMSE {rmse.value:.4f} <= 0.04", rmse.value <= CRIT["C"]), ("D", f"|bias| {abs(bias.value):.4f} <= 0.01", abs(bias.value) <= CRIT["D"]),
      ("E", f"worst year {worst_year:.3f}, worst province {worst_prov:.3f} >= 0", min(worst_year, worst_prov) >= 0),
      ("F", f"90% coverage 2021-2022 {cov:.3f} in [0.87, 0.93]", CRIT["F"][0] <= cov <= CRIT["F"][1])]
vs = CI.loc[("v3", "skill vs v1")]
R2 = {"skill_vs_v1": float(vs.value), "ci": [float(vs.ci_low), float(vs.ci_high)], "pass": bool(vs.ci_high >= 0)}
DEC = {"R1": [{"criterion": a, "result": b, "pass": bool(c)} for a, b, c in R1], "R1_pass": all(c for _, _, c in R1), "R2": R2}
DEC["usable_for_simulations"] = DEC["R1_pass"] and R2["pass"]
display(CI.round(4)); print(json.dumps(DEC, indent=1))

value  ci_low  ci_high
model       metric                                       
v3          MAE                   0.0060  0.0051   0.0069
            RMSE                  0.0099  0.0087   0.0110
            R2                    0.9913  0.9886   0.9932
            bias                 -0.0013 -0.0020  -0.0007
            skill vs v1           0.2115  0.1750   0.2479
            skill vs climatology  0.6882  0.6572   0.7178
            skill vs persistence  0.6207  0.5922   0.6441
v1          MAE                   0.0073  0.0061   0.0084
            RMSE                  0.0125  0.0110   0.0139
            R2                    0.9860  0.9813   0.9895
            bias                 -0.0004 -0.0009   0.0000
climatology MAE                   0.0212  0.0189   0.0236
            RMSE                  0.0317  0.0280   0.0352
            R2                    0.9106  0.8800   0.9322
            bias                  0.0074  0.0028   0.0115
persistence MAE                   0.0139  0.0106   0.0175
            RMSE                  0.0260  0.0222   0.0294
            R2                    0.9396  0.9223   0.9539
            bias                  0.0002  0.0001   0.0003

{
 "R1": [
  {
   "criterion": "A",
   "result": "skill vs persistence 0.621 >= 0.20",
   "pass": true
  },
  {
   "criterion": "B",
   "result": "lower CI 0.592 > 0",
   "pass": true
  },
  {
   "criterion": "C",
   "result": "RMSE 0.0099 <= 0.04",
   "pass": true
  },
  {
   "criterion": "D",
   "result": "|bias| 0.0013 <= 0.01",
   "pass": true
  },
  {
   "criterion": "E",
   "result": "worst year 0.559, worst province 0.438 >= 0",
   "pass": true
  },
  {
   "criterion": "F",
   "result": "90% coverage 2021-2022 0.884 in [0.87, 0.93]",
   "pass": true
  }
 ],
 "R1_pass": true,
 "R2": {
  "skill_vs_v1": 0.21145888601257545,
  "ci": [
   0.1749610722622852,
   0.24793410345163963
  ],
  "pass": true
 },
 "usable_for_simulations": true
}


## 4. Second look at 2023–2025 (R5, report only)

In [4]:
d5, y5, p5 = rows("testing")
TCI = E.block_bootstrap(y5, {k: p5[k] for k in ("v3", "v1")}, d5.block.to_numpy(), "v3", N_BOOT, SEED).set_index(["model", "metric"])
mar = ((d5.year == 2023) & (d5.month == 3)).to_numpy()
seas5 = np.where(d5.month <= 4, "wet", "dry")
half_all = {s: E.conformal_halfwidth(res[seas == s], NOMINAL) for s in ("wet", "dry")}
R5 = {"rmse_v3": float(TCI.loc[("v3", "RMSE"), "value"]), "rmse_v1": float(TCI.loc[("v1", "RMSE"), "value"]),
      "skill_vs_v1": [float(TCI.loc[("v3", "skill vs v1"), x]) for x in ("value", "ci_low", "ci_high")],
      "march_2023_rmse": {k: E.regression_metrics(y5[mar], p5[k][mar])["RMSE"] for k in ("v3", "v1")},
      "march_2023_bias": {k: E.regression_metrics(y5[mar], p5[k][mar])["bias"] for k in ("v3", "v1")},
      "coverage_90_by_season": float((np.abs(p5["v3"] - y5) <= np.vectorize(half_all.get)(seas5)).mean())}
gain = V3m.get_booster().get_score(importance_type="total_gain"); tot = sum(gain.values())
R5["gain_share"] = {"last_month_root_soil": gain.get("sm_root_m3m3_lag1", 0) / tot, "surface_soil": gain.get("sm_surface_m3m3_lag1", 0) / tot,
                    "chirps_rain_4": sum(gain.get(f, 0) for f in F.SM_FEATURES if f.startswith("rain")) / tot,
                    "era5_rain_4": sum(gain.get(f, 0) for f in R.FEATURES) / tot}
print(json.dumps(R5, indent=1))

{
 "rmse_v3": 0.014220358930345282,
 "rmse_v1": 0.019796127728789605,
 "skill_vs_v1": [
  0.28165956872138453,
  0.2414072799059457,
  0.3216513944396066
 ],
 "march_2023_rmse": {
  "v3": 0.024479795819028604,
  "v1": 0.05325307922604079
 },
 "march_2023_bias": {
  "v3": -0.006964947897158783,
  "v1": -0.036725163519284426
 },
 "coverage_90_by_season": 0.8216481836923296,
 "gain_share": {
  "last_month_root_soil": 0.6480145922916525,
  "surface_soil": 0.2221613023987242,
  "chirps_rain_4": 0.0029732953925603925,
  "era5_rain_4": 0.07996120906132223
 }
}


## 5. Projection hindcasts for v3 (R4)

In [5]:
CUBE = P.build_cube(MP, QP)
VALID = ~np.isnan(CUBE.v["sm_root_m3m3"]).all(axis=1); INSIDE = (CUBE.cells.province != "outside").to_numpy() & VALID
YEARS = CUBE.dates.year.to_numpy(); POOL, UNTIL = list(range(1998, 2018)), "2017-12-01"
CLIM = {m: np.nanmean(CUBE.v["sm_root_m3m3"][:, (CUBE.dates.month == m) & (YEARS <= 2017)], axis=1) for m in range(1, 13)}
def hindcast(starts, last):
    out = []
    for s in starts:
        h = min(12, (last.year - s.year) * 12 + last.month - s.month)
        if h < 1: continue
        run = P.project_sm(V3, CUBE, s, h, POOL, until=UNTIL)
        lo, med, hi = np.quantile(run["members"], [0.05, 0.5, 0.95], axis=0); pers = CUBE.v["sm_root_m3m3"][:, CUBE.t(s)]
        for k, dd in enumerate(run["dates"]):
            yy = CUBE.v["sm_root_m3m3"][:, CUBE.t(dd)]; ok = VALID & ~np.isnan(yy)
            out.append(pd.DataFrame({"lead": k + 1, "block": CUBE.cells.block[ok], "y": yy[ok], "median": med[ok, k], "lo": lo[ok, k], "hi": hi[ok, k],
                                     "climatology": CLIM[dd.month][ok], "persistence": pers[ok]}))
    return pd.concat(out, ignore_index=True)
HV = hindcast(pd.date_range("2017-12-01", "2022-11-01", freq="MS"), pd.Timestamp("2022-12-01"))
HT = hindcast(pd.date_range("2022-12-01", "2025-11-01", freq="MS"), pd.Timestamp("2025-12-01"))
def skill(H):
    rows_ = []
    for L, g in H.groupby("lead"):
        t = E.block_bootstrap(g.y.to_numpy(), {"projection": g["median"].to_numpy(), "climatology": g.climatology.to_numpy(), "persistence": g.persistence.to_numpy()},
                              g.block.to_numpy(), "projection", N_BOOT, SEED).set_index(["model", "metric"])
        bestr = "climatology" if t.loc[("climatology", "RMSE"), "value"] < t.loc[("persistence", "RMSE"), "value"] else "persistence"
        s_ = t.loc[("projection", f"skill vs {bestr}")]
        rows_.append({"lead": int(L), "best_rule": bestr, "skill": float(s_.value), "skill_ci_low": float(s_.ci_low), "skill_ci_high": float(s_.ci_high)})
    return pd.DataFrame(rows_)
SKV, SKT = skill(HV), skill(HT)
useful = 0
for _, r in SKV.iterrows():
    if r.skill_ci_low > 0: useful = int(r.lead)
    else: break
sc = np.maximum(HV.lo - HV.y, HV.y - HV.hi)
CORR = {int(L): float(np.sort(v)[min(len(v), math.ceil((len(v) + 1) * NOMINAL)) - 1]) for L, v in sc.groupby(HV.lead)}
c = HT.lead.map(CORR); cov_t = float(((HT.y >= HT.lo - c) & (HT.y <= HT.hi + c)).mean())
R4 = {"useful_lead_months": useful, "correction_by_lead": CORR, "coverage_90_2023_2025": cov_t}
display(SKV.round(3)); print(json.dumps(R4, indent=1))

,lead,best_rule,skill,skill_ci_low,skill_ci_high
0,1,persistence,0.432,0.394,0.468
1,2,climatology,0.173,0.116,0.242
2,3,climatology,0.088,0.046,0.141
3,4,climatology,0.065,0.025,0.119
4,5,climatology,0.053,0.015,0.105
5,6,climatology,0.039,0.001,0.090
6,7,climatology,0.025,-0.014,0.077
7,8,climatology,0.015,-0.024,0.067
8,9,climatology,0.009,-0.029,0.060
9,10,climatology,0.007,-0.030,0.056


{
 "useful_lead_months": 6,
 "correction_by_lead": {
  "1": 0.004788623336288669,
  "2": 0.006226680281259056,
  "3": 0.007422541242132902,
  "4": 0.008063175444029735,
  "5": 0.007995043422586223,
  "6": 0.007981821574190584,
  "7": 0.0074064676399512586,
  "8": 0.00686578513473593,
  "9": 0.006109623242450013,
  "10": 0.00535496502153332,
  "11": 0.005016657579627493,
  "12": 0.00464557467769551
 },
 "coverage_90_2023_2025": 0.8068141691926343
}


## 6. Scenario checks S1–S4 with v3 (R3; same rules as docs/ML_API_VALIDATION.md)

In [6]:
TOL, STRICT = V1["card"]["metrics"]["validation_2018_2022"]["RMSE"]["value"], 1e-4
def cut(w, h): return {k: (v[:, :, :h] if isinstance(v, np.ndarray) and v.ndim == 3 else v) for k, v in w.items()}
def med(w, s, h): return np.median(P.project_sm(V3, CUBE, s, h, None, weather=cut(w, h))["members"][:, INSIDE, :], axis=0)
RUNS = []
for s in pd.date_range("2017-12-01", "2021-12-01", freq="MS"):
    w = P.analog_weather(CUBE, s, 12, POOL, until=UNTIL)
    if len(w["years"]) < 5: continue
    r = {"base": med(w, s, 12), "t_plus": med(P.scenario_adjust(w, t2m_delta=2.0), s, 3), "t_minus": med(P.scenario_adjust(w, t2m_delta=-2.0), s, 3),
         "rain_half": med(P.scenario_adjust(w, rain_factor=0.5), s, 3), "rain_double": med(P.scenario_adjust(w, rain_factor=2.0), s, 3)}
    for n in range(1, 7): r[f"dry{n}"] = med(P.scenario_drought(w, list(range(1, n + 1)), CUBE, s), s, 12)
    RUNS.append(r)
stack = lambda key, lead: np.concatenate([r[key][:, lead - 1] - r["base"][:, lead - 1] for r in RUNS])
CH = {}
dp, dm = stack("t_plus", 3), stack("t_minus", 3)
CH["S1"] = {"raise_share_plus2": float((dp > TOL).mean()), "fall_share_minus2": float((dm < -TOL).mean()), "strict_raise_share_plus2": float((dp > STRICT).mean()),
            "mean_effect_plus2": float(dp.mean()), "mean_effect_minus2": float(dm.mean())}
CH["S1"]["pass"] = CH["S1"]["raise_share_plus2"] <= 0.01 and CH["S1"]["fall_share_minus2"] <= 0.01
tab = []
for n in range(1, 7):
    e, l6 = stack(f"dry{n}", n), stack(f"dry{n}", 6)
    tab.append({"months": n, "mean_change_end": float(e.mean()), "mean_change_lead6": float(l6.mean()), "rise_share_end": float((e > TOL).mean())})
S2 = pd.DataFrame(tab); mono = all(S2.mean_change_lead6.iloc[i + 1] <= S2.mean_change_lead6.iloc[i] + 0.001 for i in range(5))
CH["S2"] = {"table": tab, "monotone_lead6": mono, "pass": bool((S2.mean_change_end < 0).all() and mono and (S2.rise_share_end <= 0.01).all())}
dh, dd = stack("rain_half", 3), stack("rain_double", 3)
CH["S3"] = {"half_raise_share": float((dh > TOL).mean()), "double_lower_share": float((dd < -TOL).mean()), "mean_effect_half": float(dh.mean()), "mean_effect_double": float(dd.mean())}
CH["S3"]["pass"] = CH["S3"]["half_raise_share"] <= 0.01 and CH["S3"]["double_lower_share"] <= 0.01 and CH["S3"]["mean_effect_half"] <= -0.003 and CH["S3"]["mean_effect_double"] >= 0.003
d3, d12 = stack("dry3", 3), stack("dry3", 12); okk = np.abs(d3) > 1e-6
CH["S4"] = {"share_forgetting": float((np.abs(d12[okk]) < np.abs(d3[okk])).mean()), "mean_change_lead3": float(d3.mean()), "mean_change_lead12": float(d12.mean())}
CH["S4"]["pass"] = CH["S4"]["share_forgetting"] >= 0.90
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "table"} for k, v in CH.items()}, indent=1)); S2.round(4)

{
 "S1": {
  "raise_share_plus2": 0.0,
  "fall_share_minus2": 0.0,
  "strict_raise_share_plus2": 0.005504971219256934,
  "mean_effect_plus2": -0.003142569449299889,
  "mean_effect_minus2": 0.003316143419104275,
  "pass": true
 },
 "S2": {
  "monotone_lead6": true,
  "pass": true
 },
 "S3": {
  "half_raise_share": 0.0,
  "double_lower_share": 0.0,
  "mean_effect_half": -0.011617483157140151,
  "mean_effect_double": 0.019008500929697685,
  "pass": true
 },
 "S4": {
  "share_forgetting": 0.7584669018729258,
  "mean_change_lead3": -0.030063652371703088,
  "mean_change_lead12": -0.007454539337344098,
  "pass": false
 }
}


,months,mean_change_end,mean_change_lead6,rise_share_end
0,1,-0.0073,-0.0045,0.0000
1,2,-0.0194,-0.0101,0.0000
2,3,-0.0301,-0.0178,0.0000
3,4,-0.0385,-0.0285,0.0000
4,5,-0.0455,-0.0438,0.0001
5,6,-0.0525,-0.0525,0.0001


## 7. Save

In [7]:
out = ART / "sm_root_monthly"; out.mkdir(exist_ok=True)
V3m.save_model(out / "model.ubj")
(out / "preprocessing.json").write_text(json.dumps({"feature_names_in_order": FEATS, "monotone_constraints": MONO, "training_feature_bounds": V3["bounds"],
                                                    "imputation": "none", "scaling": "none", "era5_rain_units": "mm per month (total_precipitation_sum x 1000)"}, indent=1))
summary = {"status": "candidate" if DEC["usable_for_simulations"] else "rejected", "decision": DEC, "second_look_2023_2025": R5, "projections": R4,
           "scenario_checks": CH, "trees": best, "hyperparameters": {k: v for k, v in params.items() if k != "monotone_constraints"},
           "sha256": D.sha256(out / "model.ubj"), "reference_v1_sha256": V1["card"]["sha256"],
           "versions": {"python": platform.python_version(), "xgboost": xgb.__version__}}
(ART / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
SKV.to_csv(ART / "hindcast_skill_validation.csv", index=False); SKT.to_csv(ART / "hindcast_skill_2023_2025.csv", index=False); S2.to_csv(ART / "s2_droughts.csv", index=False)
print(sorted(p.name for p in ART.rglob("*") if p.is_file()))

['hindcast_skill_2023_2025.csv', 'hindcast_skill_validation.csv', 'model.ubj', 'preprocessing.json', 's2_droughts.csv', 'summary.json']
